In [3]:
import arcpy
import os

out_gdb = r'C:\Users\colet\Documents\GEOG5890_Capstone\GEOG5890_Capstone_GDB\GeoAI_Capstone.gdb'

crash_grid = os.path.join(out_gdb, 'Crash_Grid')
monthly_table = os.path.join(out_gdb, 'Monthly_Crash_Counts')

count_field = 'COUNT_OBJECTID'

complete_table = os.path.join(out_gdb, 'Complete_Grid_Monthly_Data')

In [4]:
crash_counts = {}

fields = ['Grid_ID', 'YearMonth', count_field]

with arcpy.da.SearchCursor(monthly_table, fields) as cursor:
    for grid_id, year_month, count in cursor:
        crash_counts[(grid_id, year_month)] = int(count or 0)

In [5]:
months = [
    f"{year}-{month:02d}"
    for year in range(2015, 2025)
    for month in range(1, 13)
]

In [6]:
arcpy.management.CreateTable(out_gdb, 'Complete_Grid_Monthly_Data')

arcpy.management.AddField(complete_table, 'GRID_ID', 'LONG')
arcpy.management.AddField(complete_table, 'YearMonth', 'TEXT', field_length = 10)
arcpy.management.AddField(complete_table, 'CrashCount', 'LONG')

<Result 'C:\\Users\\colet\\Documents\\GEOG5890_Capstone\\GEOG5890_Capstone_GDB\\GeoAI_Capstone.gdb\\Complete_Grid_Monthly_Data'>

In [8]:
with arcpy.da.SearchCursor(crash_grid, ["Grid_ID"]) as grid_cursor:
    grid_ids = [row[0] for row in grid_cursor]

with arcpy.da.InsertCursor(complete_table, ['GRID_ID', 'YearMonth', 'CrashCount']) as insert_cursor:
    for grid_id in grid_ids:
        for year_month in months:
            count = crash_counts.get((grid_id, year_month), 0)
            insert_cursor.insertRow([grid_id, year_month, count])